# Chapter 9: The Masterpiece
## Companion Notebook — "Eyes of the Machine"

Build a real Computer Vision project: a Smart Face Detection Filter.

---
### 9A: Setup — Import and Load Face Cascade
OpenCV comes with pre-trained face detectors.

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
from google.colab import files

face_cascade = cv.CascadeClassifier(
    cv.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

print("Face cascade loaded!")

---
### 9B: Load a Photo with Faces

In [ ]:
print("Upload a photo with faces (group photo works best):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

img_bgr = cv.imread(filename)
img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
img_gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

print(f"Image loaded: {img_rgb.shape}")

---
### 9C: Detect Faces — The Magic Line
`detectMultiScale` finds faces at all sizes.

In [ ]:
faces = face_cascade.detectMultiScale(
    img_gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30, 30)
)

print(f"Detected {len(faces)} face(s) in the image.")

---
### 9D: Draw Rectangles Around Faces
Visual confirmation that detection worked.

In [ ]:
img_with_boxes = img_rgb.copy()

for (x, y, w, h) in faces:
    cv.rectangle(img_with_boxes, (x, y), (x + w, y + h), (0, 255, 0), 3)

plt.figure(figsize=(10, 8))
plt.imshow(img_with_boxes)
plt.title(f"Detected {len(faces)} Face(s)")
plt.axis('off')
plt.show()

---
### 9E: Privacy Filter — Blur Faces Automatically
Used by news agencies and Google Maps to hide identities.

In [ ]:
img_filtered = img_rgb.copy()

for (x, y, w, h) in faces:
    face_region = img_filtered[y:y+h, x:x+w]
    blurred_face = cv.GaussianBlur(face_region, (99, 99), 30)
    img_filtered[y:y+h, x:x+w] = blurred_face

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img_filtered)
plt.title("Privacy Mode: Faces Blurred")
plt.axis('off')
plt.show()

---
### 9F: Fun Filter — Sunglasses Overlay
Like Snapchat filters — drawn with basic shapes.

In [ ]:
img_sunglasses = img_rgb.copy()

for (x, y, w, h) in faces:
    glasses_y = y + h // 4
    glasses_height = h // 6

    cv.rectangle(img_sunglasses, (x + 5, glasses_y), (x + w // 2 - 5, glasses_y + glasses_height), (0, 0, 0), -1)
    cv.rectangle(img_sunglasses, (x + w // 2 + 5, glasses_y), (x + w - 5, glasses_y + glasses_height), (0, 0, 0), -1)
    cv.rectangle(img_sunglasses, (x + w // 2 - 5, glasses_y + glasses_height // 3), (x + w // 2 + 5, glasses_y + 2 * glasses_height // 3), (0, 0, 0), -1)

plt.figure(figsize=(10, 8))
plt.imshow(img_sunglasses)
plt.title("Smart Filter: Sunglasses Overlay")
plt.axis('off')
plt.show()

---
### 9H: The Complete Smart Filter Function
A reusable function that applies any effect to detected faces.

In [ ]:
def smart_filter(image_path, effect='blur'):
    img_bgr = cv.imread(image_path)
    img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
    img_gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(img_gray, 1.1, 5, minSize=(30, 30))
    result = img_rgb.copy()

    for (x, y, w, h) in faces:
        if effect == 'blur':
            face_region = result[y:y+h, x:x+w]
            blurred_face = cv.GaussianBlur(face_region, (99, 99), 30)
            result[y:y+h, x:x+w] = blurred_face
        elif effect == 'sunglasses':
            glasses_y = y + h // 4
            glasses_height = h // 6
            cv.rectangle(result, (x + 5, glasses_y), (x + w // 2 - 5, glasses_y + glasses_height), (0, 0, 0), -1)
            cv.rectangle(result, (x + w // 2 + 5, glasses_y), (x + w - 5, glasses_y + glasses_height), (0, 0, 0), -1)
            cv.rectangle(result, (x + w // 2 - 5, glasses_y + glasses_height // 3), (x + w // 2 + 5, glasses_y + 2 * glasses_height // 3), (0, 0, 0), -1)
        else:
            cv.rectangle(result, (x, y), (x + w, y + h), (0, 255, 0), 3)

    return result, len(faces)

# Test it.
result_img, num_faces = smart_filter(filename, effect='blur')
print(f"Smart filter applied! Detected {num_faces} face(s).")

plt.figure(figsize=(10, 8))
plt.imshow(result_img)
plt.title(f"Smart Filter: {num_faces} face(s) detected and blurred")
plt.axis('off')
plt.show()

cv.imwrite("smart_filter_result.jpg", cv.cvtColor(result_img, cv.COLOR_RGB2BGR))
print("Result saved as smart_filter_result.jpg")

---
### 9I: Compare All Three Effects

In [ ]:
effects = ['box', 'blur', 'sunglasses']
plt.figure(figsize=(15, 5))

for i, effect in enumerate(effects):
    result_img, num_faces = smart_filter(filename, effect=effect)
    plt.subplot(1, 3, i + 1)
    plt.imshow(result_img)
    plt.title(f"Effect: {effect}\n({num_faces} face(s))")
    plt.axis('off')

plt.tight_layout()
plt.show()

---
## 🧠 Challenge: The "Census Taker"

Use a crowd photo. Detect faces and:
1. Give each face a random colored box.
2. Number each face.
3. Print total count:
   "The AI census reports [X] humans detected."

In [ ]:
img_census = img_rgb.copy()

for i, (x, y, w, h) in enumerate(faces):
    color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
    cv.rectangle(img_census, (x, y), (x + w, y + h), color, 3)
    cv.putText(img_census, str(i + 1), (x, y - 10),
               cv.FONT_HERSHEY_SIMPLEX, 1, color, 2)

plt.imshow(img_census)
plt.title(f"Census: {len(faces)} face(s) detected")
plt.axis('off')
plt.show()

print(f"The AI census reports {len(faces)} humans detected. Please verify.")

---
## 🚀 Next Steps (Beyond This Book)

1. **Smile detection:** Use `haarcascade_smile.xml` instead.
2. **Eye detection:** Use `haarcascade_eye.xml`.
3. **Profile faces:** Use `haarcascade_profileface.xml`.
4. **Real-time webcam:** Run the local Python script from the chapter.
5. **Face recognition:** Use `face_recognition` library to IDENTIFY who each person is.

You have all the tools. Go build.